# Terragrunt blocks

In [3]:
from bs4 import BeautifulSoup, NavigableString
from os import path, environ
from IPython.display import display
from IPython.core.display import HTML
import re
import pandas as pd

assets_abspath = path.join(environ.get("ASSETS_ABSPATH", "-"), "terragrunt")
artifacts_abspath = path.join(
    environ.get("ARTIFACTS_ABSPATH", "-"), "terragrunt"
)
blocks_source_abspath = path.join(assets_abspath, "blocks.html")
blocks_target_abspath = path.join(artifacts_abspath, "blocks.csv")

In [4]:
with open(blocks_source_abspath, "r") as f:
    html = f.read()

In [5]:
DOMAIN = "https://terragrunt.gruntwork.io"
ROUTE = "/docs/reference/config-blocks-and-attributes"

soup = BeautifulSoup(html, "html.parser")
# Remove classes and styles
for elem in soup.descendants:
    if hasattr(elem, "class"):
        del elem["class"]
    if hasattr(elem, "style"):
        del elem["style"]

# Fix link bases
for anchor in soup.find_all("a"):
    if anchor["href"].startswith("#"):
        anchor["href"] = f"{DOMAIN}{ROUTE}{anchor['href']}"
    if anchor["href"].startswith("/"):
        anchor["href"] = f"{DOMAIN}{anchor['href']}"

# Split content
entries = []
current = None
for elem in soup.children:
    if isinstance(elem, NavigableString):
        continue
    if elem.name == "h3":
        if current is not None:
            entries.append(current)
        current = {
            "header_h3_soup": elem,
            "description_soup": BeautifulSoup(
                f"<h2>{elem.text}</h2>", "html.parser"
            ),
        }
        continue
    if current is not None:
        current["description_soup"].append(elem)
entries.append(current)

# Enrich entries
enriched = []
for entry in entries:
    header_text = entry["header_h3_soup"].text.strip()
    summary_text = entry["description_soup"].p.text.strip()
    summary_text = re.split(r"\.\s", summary_text)[0] + "."
    summary_text = re.sub(header_text, "___", summary_text)
    enriched.append(
        {
            **entry,
            "header_p_soup": BeautifulSoup(
                f"<p>{header_text}</p>", "html.parser"
            ),
            "summary_soup": BeautifulSoup(
                f"<p>{summary_text}</p>", "html.parser"
            ),
        }
    )

htmlful = []
for item in enriched:
    htmlful.append(
        {
            "header_p_html": item["header_p_soup"].prettify(),
            "summary_html": item["summary_soup"].prettify(),
            "description_html": item["description_soup"].prettify(),
        }
    )

In [6]:
df = pd.DataFrame(htmlful)
df.rename(
    columns={
        "header_p_html": "HeaderHtml",
        "summary_html": "SummaryHtml",
        "description_html": "DescriptionHtml",
    },
    inplace=True,
)
df["Tags"] = "WebScraped Terragrunt-v0.62.1"
df

,HeaderHtml,SummaryHtml,DescriptionHtml,Tags
0,<p>\n terraform\n</p>\n,<p>\n The ___ block is used to configure how T...,<h2>\n terraform\n</h2>\n<p>\n The\n <code>\n ...,WebScraped Terragrunt-v0.62.1
1,<p>\n remote_state\n</p>\n,<p>\n The ___ block is used to configure how\n...,<h2>\n remote_state\n</h2>\n<p>\n The\n <code>...,WebScraped Terragrunt-v0.62.1
2,<p>\n include\n</p>\n,<p>\n The ___ block is used to specify inherit...,<h2>\n include\n</h2>\n<p>\n The\n <code>\n i...,WebScraped Terragrunt-v0.62.1
3,<p>\n locals\n</p>\n,<p>\n The ___ block is used to define aliases ...,<h2>\n locals\n</h2>\n<p>\n The\n <code>\n lo...,WebScraped Terragrunt-v0.62.1
4,<p>\n dependency\n</p>\n,<p>\n The ___ block is used to configure modul...,<h2>\n dependency\n</h2>\n<p>\n The\n <code>\n...,WebScraped Terragrunt-v0.62.1
5,<p>\n dependencies\n</p>\n,<p>\n The ___ block is used to enumerate all t...,<h2>\n dependencies\n</h2>\n<p>\n The\n <code>...,WebScraped Terragrunt-v0.62.1
6,<p>\n generate\n</p>\n,<p>\n The ___ block can be used to arbitrarily...,<h2>\n generate\n</h2>\n<p>\n The\n <code>\n ...,WebScraped Terragrunt-v0.62.1


In [7]:
df.to_csv(blocks_target_abspath, sep="|", header=False, index=False)